In [1]:
import os
from matplotlib import pyplot as plt
import numpy as np
import cv2
import pickle
from glob import glob
from PIL import Image

In [2]:
att_dir = './log_mri2proto_slim_conv/spatial_attention/'
img_dir = '/mnt/data/UserData/shichen/data/NSD_dataset/subj01/training_split/training_images/'
save_dir = './log_mri2proto_slim_conv/spatial_attention/result'
os.makedirs(save_dir, exist_ok=True)
# read the attention data into dictionary
all_area = glob(os.path.join(att_dir, '*.pkl'))
att_data = dict()
for cur_area in all_area:
    cur_data = pickle.load(open(cur_area, 'rb'))
    att_data[os.path.basename(cur_area)[:-4]] = cur_data

In [3]:
def visualize_attention_maps(att_data, img_dir, save_dir, alpha=0.5, sigma=5):
    """
    att_data: dict[category][sample_name] -> HxW attention map
    img_dir: directory containing raw images
    save_dir: directory to save visualizations
    alpha: transparency for overlay
    sigma: gaussian smoothing strength
    """

    os.makedirs(save_dir, exist_ok=True)

    categories = list(att_data.keys())
    k = len(categories)

    sample_names = list(att_data[categories[0]].keys())

    # Build lookup for images (basename -> path)
    img_lookup = {}
    for fname in os.listdir(img_dir):
        base = os.path.splitext(fname)[0]
        img_lookup[base] = os.path.join(img_dir, fname)

    for sample in sample_names:

        if sample not in img_lookup:
            print(f"Image not found for sample: {sample}")
            continue

        img = np.array(Image.open(img_lookup[sample]).convert("RGB"))
        H, W = img.shape[:2]

        fig, axes = plt.subplots(1, k, figsize=(4 * k, 4))
        if k == 1:
            axes = [axes]

        for i, cat in enumerate(categories):

            att = att_data[cat][sample]

            # resize attention map
            att_resized = cv2.resize(att, (W, H), interpolation=cv2.INTER_LINEAR)

            # gaussian smoothing
            att_smoothed = cv2.GaussianBlur(att_resized, (0, 0), sigmaX=sigma, sigmaY=sigma)

            # normalize for visualization
            att_smoothed -= att_smoothed.min()
            if att_smoothed.max() > 0:
                att_smoothed /= att_smoothed.max()

            ax = axes[i]
            ax.imshow(img)
            ax.imshow(att_smoothed, cmap="jet", alpha=alpha)
            ax.set_title(cat)
            ax.axis("off")

        plt.tight_layout()

        save_path = os.path.join(save_dir, f"{sample}.png")
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        plt.close(fig)


In [4]:
visualize_attention_maps(att_data, img_dir, save_dir, sigma=15)